In [1]:
import os
import pandas as pd
import numpy as np
from pathlib import Path
  
base_dir = str(Path.cwd().parent)
methods = ["Delphos", "SA", "VNS"]
seeds = [f"seed_{i}" for i in range(101, 111)]
budget = f"B_{1500}"

# --- 1: Collect Data ---
per_seed_data = []          
all_valid_models = {m: [] for m in methods}  

for method in methods:
    for seed in seeds:        
        filename = "models_new.csv" if method == "Delphos" else "estimations.csv"
        file_path = os.path.join(base_dir, method, budget, seed, filename)                 
        if not os.path.exists(file_path):
            print(f"File not found: {file_path}")
            continue
                
        try:
            if method == "Delphos":
                usecols = ['successfulEstimation', 'numParams', 'LLout', 'AIC', 'BIC', 'timeTaken']
                df = pd.read_csv(file_path, usecols=usecols)
                df = df.rename(columns={'LLout': 'LogLikelihood', 'timeTaken': 'TimeTaken'})
                    
            elif method == "SA":
                usecols = ['valid', 'log_likelihood', 'bic', 'aic', 'seconds', 'parameters']
                df = pd.read_csv(file_path, usecols=usecols)
                df = df.rename(columns={
                    'aic': 'AIC', 'bic': 'BIC', 'parameters': 'numParams',
                    'log_likelihood': 'LogLikelihood', 'seconds': 'TimeTaken'
                })
                    
            elif method == "VNS":
                usecols = ['valid', 'log_likelihood', 'parameters', 'bic', 'seconds']
                df = pd.read_csv(file_path, usecols=usecols)
                df['AIC'] = 2 * df['parameters'] - 2 * df['log_likelihood']
                df = df.rename(columns={
                    'bic': 'BIC', 'parameters': 'numParams',
                    'log_likelihood': 'LogLikelihood', 'seconds': 'TimeTaken'
                })
            
            # Total models attempted in this seed
            total_models = len(df)
                
            if 'successfulEstimation' in df.columns:
                df_valid = df[df['successfulEstimation'] == True].copy()
            elif 'valid' in df.columns:
                df_valid = df[df['valid'] == True].copy()
            else:
                df_valid = df.copy()
            
            df_valid = df_valid.dropna(subset=['AIC'])
            df_valid = df_valid[np.isfinite(df_valid['AIC']) & np.isfinite(df_valid['numParams'])]
                
            # Append all valid models for global stats
            all_valid_models[method].append(df_valid[['AIC', 'numParams']])
            
            # Per-seed stats
            total_time = df['TimeTaken'].sum()
            num_valid = len(df_valid)
                
            per_seed_data.append({
                'Method': method,
                'Seed': seed,
                'Total_Time (s)': total_time,
                'Converged_Models': num_valid,
                'Total_Models': total_models,
            })
                
        except Exception as e:
            print(f"Error processing {file_path}: {e}")

print("Data successfully loaded!")

# --- 2: Compute Statistics ---
print("\n" + "=" * 70)
print("                    SUMMARY STATISTICS (B_1500)                       ")
print("=" * 70)

for method in methods:
    # -- Global stats (across ALL valid models, regardless of seed) --
    pooled_df = pd.concat(all_valid_models[method], ignore_index=True)
    
    mean_aic = pooled_df['AIC'].mean()
    std_aic  = pooled_df['AIC'].std()
    
    mean_params = pooled_df['numParams'].mean()
    std_params  = pooled_df['numParams'].std()
    
    total_valid = len(pooled_df)
    
    # -- Per-seed stats --
    seed_df = pd.DataFrame([r for r in per_seed_data if r['Method'] == method])
    
    mean_time = seed_df['Total_Time (s)'].mean()
    std_time  = seed_df['Total_Time (s)'].std()
    
    mean_converged = seed_df['Converged_Models'].mean()
    std_converged  = seed_df['Converged_Models'].std()
    
    mean_total = seed_df['Total_Models'].mean()
    
    print(f"\n--- {method} ---")
    print(f"  AIC (all valid models)         : {mean_aic:>10.2f} (±{std_aic:.2f})")
    print(f"  Num Params (all valid models)  : {mean_params:>10.2f} (±{std_params:.2f})")
    print(f"  Time per seed (s)              : {mean_time:>10.2f} (±{std_time:.2f})")
    print(f"  Time per seed (min)            : {mean_time/60:>10.2f} (±{std_time/60:.2f})")
    print(f"  Converged models per seed      : {mean_converged:>10.1f} (±{std_converged:.1f})  out of {mean_total:.0f}")
    print(f"  Total valid models (all seeds) : {total_valid}")
    print("-" * 70)


Data successfully loaded!

                    SUMMARY STATISTICS (B_1500)                       

--- Delphos ---
  AIC (all valid models)         :    7957.80 (±394.52)
  Num Params (all valid models)  :      21.66 (±7.81)
  Time per seed (s)              :    1421.00 (±216.04)
  Time per seed (min)            :      23.68 (±3.60)
  Converged models per seed      :      645.3 (±64.8)  out of 1500
  Total valid models (all seeds) : 6453
----------------------------------------------------------------------

--- SA ---
  AIC (all valid models)         :    8341.06 (±472.26)
  Num Params (all valid models)  :      12.75 (±6.55)
  Time per seed (s)              :     544.66 (±97.54)
  Time per seed (min)            :       9.08 (±1.63)
  Converged models per seed      :     1384.4 (±20.5)  out of 1500
  Total valid models (all seeds) : 13844
----------------------------------------------------------------------

--- VNS ---
  AIC (all valid models)         :    8843.86 (±385.10)
  Num Pa